$\text{Binary Cross Entropy} (\text{concatenated logits}~(X^{`}_{0})~\forall~\text{30 discriminator heads}, ones)$

- $d_{\text{loss}} = loss_{\text{real}} + loss_{\text{real}}$
            - $ loss_{\text{real}} = Binary Cross Entropy (\text{concatenated logits}~(X_{0})~\forall~\text{30 discriminator heads}, ones)$
            - $ loss_{\text{fake}} = Binary Cross Entropy (\text{concatenated logits}(X_{0})~\forall~\text{30 discriminator heads}, zeros)$

# 3. Notes on training and adversarial supervison:
- *Discriminators* : Simple conv net discriminator heads across all 30 transformer layers of the pre-trained teacher model, as advised in the LADD paper.
- *Inputs to the discriminator heads* : 
    - Z-image backbone runs unified attention over concatenated image-text embeddings.
    - The forward pass of ```ZImageTransformer2DModel``` is modified to capture concatenated activations after every transformer block (Refer forward in ```VideoX-Fun/videox-fun/models/z_image_transformer2d.py```)
    - The activations $(B, img-len+cap-len, 3840)$ tensors are first sliced to extract only the vision part of the stream then reshaped to be processed by the conv net discriminators to yield logits.
        - ![alt text](image-8.png)
- Adversarial loss:
    - Generator loss: $\text{Binary Cross Entropy} (\text{concatenated logits}~(X^{`}_{0})~\forall~\text{30 discriminator heads}, ones)$
        - Incentivize teacher activations corresponding to $X^{`}_{0}$ prediction from generator to look as if they are similar to those corresponding to synthetic $X_{0} \sim p_{\text{data}}$
        - Fool the discriminator.
    - Discriminator loss:
        - $d_{\text{loss}} = loss_{\text{real}} + loss_{\text{real}}$
            - $ loss_{\text{real}} = \text{Binary Cross Entropy} (\text{concatenated logits}~(X_{0})~\forall~\text{30 discriminator heads}, ones)$
            - $ loss_{\text{fake}} = \text{Binary Cross Entropy} (\text{concatenated logits}(X_{0})~\forall~\text{30 discriminator heads}, zeros)$
            - Discriminator is incentivized to separate logits for reals and fakes.
- Reconstruction Loss (distill_loss):
    - Regresses $x^{`}_{0}$ from student against $x_{0} \sim p_{\text{data}}$
- Total Generator Loss = recon_loss + Generator Loss
- Time step sampling:
    - Discrete time steps $\in [1.0, 0.75, 0.5, 0.25]$ are sampled according to the annealing schedule defined in the paper.
        - ![alt text](image-9.png)
    - Discriminators are also conditioned on the discrete time steps using simple additive modulation from ```nn.Sequential(nn.Linear(1, t_hidden_dim), nn.Silu(), nn.Linear(t_hiiden_dim, hidden_channels))``` (refer CondFeatureDiscHead class in VideoX-Fun/scrips/z_image/train.py)
- Training alternates between updating Generator and then Discriminator at each step. On wandb dashboard one step corresponds to one generator and one discriminator update.